<a href="https://colab.research.google.com/github/VighneshGaya/ALS-transcriptomics-analysis/blob/main/ALS_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas numpy matplotlib seaborn scipy statsmodels scikit-learn gseapy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.0/596.0 kB 9.3 MB/s eta 0:00:00


In [21]:
gse76220_expr_url = "https://raw.githubusercontent.com/VighneshGaya/ALS-transcriptomics-analysis/main/data/GSE76220/GSE76220_ALS_LCM_MPR_gene_expression.txt.gz"
gse76220_rpkm_url = "https://raw.githubusercontent.com/VighneshGaya/ALS-transcriptomics-analysis/main/data/GSE76220/GSE76220_ALS_LCM_RPKM.txt.gz"
gse182976_counts_url = "https://raw.githubusercontent.com/VighneshGaya/ALS-transcriptomics-analysis/main/data/GSE182976/GSE182976_norm_counts_TPM_GRCh38.p13_NCBI.tsv.gz"


In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import gseapy as gp

In [22]:
expr_76220 = pd.read_csv(gse76220_expr_url, sep="\t", compression="gzip")
rpkm_76220 = pd.read_csv(gse76220_rpkm_url, sep="\t", compression="gzip")
counts_182976 = pd.read_csv(gse182976_counts_url, sep="\t", compression="gzip")

print("GSE76220 expression shape:", expr_76220.shape)
print("GSE76220 RPKM shape:", rpkm_76220.shape)
print("GSE182976 counts shape:", counts_182976.shape)

GSE76220 expression shape: (14635, 12)
GSE76220 RPKM shape: (25856, 25)
GSE182976 counts shape: (39376, 13)


In [23]:
print("GSE76220 columns:")
print(expr_76220.columns.tolist())

print("\nFirst rows of GSE76220:")
display(expr_76220.head())

print("\nLast rows of GSE76220:")
display(expr_76220.tail())

print("\nFirst rows of GSE182976:")
display(counts_182976.head())

print("\nLast rows of GSE182976:")
display(counts_182976.tail())

GSE76220 columns:
['ENS ID', 'Ratio MPR', 'Zscore MPR', 'Annotation', 'gene_ID', 'gene_symbol', 'Refseq', 'Protein_ID', 'gene name', 'Ratio MPR.1', 'Unnamed: 10', 'Unnamed: 11']

First rows of GSE76220:


,ENS ID,Ratio MPR,Zscore MPR,Annotation,gene_ID,gene_symbol,Refseq,Protein_ID,gene name,Ratio MPR.1,Unnamed: 10,Unnamed: 11
0,6263,1.000000,1.000000,6263,AX748157,AX748157,NaN,NaN,"Homo sapiens cDNA FLJ36163 fis, clone TESTI202...",1.000000,Downregulated by MPR,NaN
1,24489,0.999742,0.999397,24489,CR598614,CR598614,NaN,NaN,full-length cDNA clone CS0DF037YH05 of Fetal b...,0.999742,Upregulated by MPR,NaN
2,21371,0.999668,0.997844,21371,BC021582,SCML4,NM_198081,NP_932347,sex comb on midleg-like 4,0.999668,NaN,NaN
3,20676,0.999630,0.999815,20676,CCDS4660,OR2H1,NM_030883,NP_112145,"olfactory receptor, family 2, subfamily H,",0.999630,NaN,NaN
4,21569,0.999483,0.998277,21569,AK123801,AK123801,NaN,NaN,"Homo sapiens cDNA FLJ41807 fis, clone NOVAR200...",0.999483,NaN,NaN



Last rows of GSE76220:


,ENS ID,Ratio MPR,Zscore MPR,Annotation,gene_ID,gene_symbol,Refseq,Protein_ID,gene name,Ratio MPR.1,Unnamed: 10,Unnamed: 11
14630,26003,0.001325,0.002193,26003,NM_021992,TMSB15A,NM_021992,NP_919305,thymosin-like 8,0.001325,NaN,NaN
14631,12326,0.001189,0.001387,12326,NM_175883,OR7D2,NM_175883,NP_787079,"olfactory receptor, family 7, subfamily D,",0.001189,NaN,NaN
14632,3711,0.001164,0.002059,3711,NM_003696,OR6A2,NM_003696,NP_003687,"olfactory receptor, family 6, subfamily A,",0.001164,NaN,NaN
14633,22267,0.000975,0.003367,22267,CR623750,CR623750,NaN,NaN,full-length cDNA clone CS0DA003YM21 of Neurobl...,0.000975,NaN,NaN
14634,5326,0.000449,0.000988,5326,AY829431,DO,NaN,NaN,Dombrock blood group carrier molecule splice v...,0.000449,NaN,NaN



First rows of GSE182976:


,GeneID,GSM5548014,GSM5548015,GSM5548016,GSM5548017,GSM5548018,GSM5548019,GSM5548020,GSM5548021,GSM5548022,GSM5548023,GSM5548024,GSM5548025
0,100287102,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,653635,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,102466751,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,107985730,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,100302278,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Last rows of GSE182976:


,GeneID,GSM5548014,GSM5548015,GSM5548016,GSM5548017,GSM5548018,GSM5548019,GSM5548020,GSM5548021,GSM5548022,GSM5548023,GSM5548024,GSM5548025
39371,4541,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
39372,4556,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
39373,4519,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
39374,4576,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
39375,4571,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [25]:
import pandas as pd

file_path = "https://raw.githubusercontent.com/VighneshGaya/ALS-transcriptomics-analysis/main/data/GSE182976/GSE182976_norm_counts_TPM_GRCh38.p13_NCBI.tsv.gz"

df = pd.read_csv(file_path, sep="\t", compression="gzip")

print(df.shape)
df.head()

(39376, 13)


,GeneID,GSM5548014,GSM5548015,GSM5548016,GSM5548017,GSM5548018,GSM5548019,GSM5548020,GSM5548021,GSM5548022,GSM5548023,GSM5548024,GSM5548025
0,100287102,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,653635,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,102466751,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,107985730,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,100302278,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [26]:
import numpy as np

df = df.rename(columns={df.columns[0]: "GeneID"})
df = df.set_index("GeneID")

df_numeric = df.select_dtypes(include=[np.number])

print(df_numeric.shape)

(39376, 12)


In [27]:
non_zero_count = (df_numeric != 0).sum().sum()
print("Total non-zero values:", non_zero_count)

Total non-zero values: 135


In [28]:
non_zero_rows = df_numeric[(df_numeric != 0).any(axis=1)]

print("Genes with non-zero expression:", non_zero_rows.shape)
non_zero_rows.head()

Genes with non-zero expression: (37, 12)


,GSM5548014,GSM5548015,GSM5548016,GSM5548017,GSM5548018,GSM5548019,GSM5548020,GSM5548021,GSM5548022,GSM5548023,GSM5548024,GSM5548025
GeneID,,,,,,,,,,,,
255738,69.65,91.65,186.9,122.1,55.12,60.4,45.64,67.09,0.00,0.0,37.23,77.54
29949,0.00,0.00,0.0,0.0,0.00,0.0,0.00,0.00,36.31,0.0,0.00,0.00
105372878,0.00,0.00,0.0,0.0,0.00,0.0,0.00,0.00,162.40,0.0,0.00,0.00
8291,133.70,131.90,179.3,152.3,52.89,173.9,109.50,48.28,209.30,170.1,125.00,148.80
200403,0.00,0.00,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.0,0.00,22.90


In [29]:
df_nonzero_only = df_numeric.replace(0, np.nan).dropna(how='all')

df_nonzero_only.head()

,GSM5548014,GSM5548015,GSM5548016,GSM5548017,GSM5548018,GSM5548019,GSM5548020,GSM5548021,GSM5548022,GSM5548023,GSM5548024,GSM5548025
GeneID,,,,,,,,,,,,
255738,69.65,91.65,186.9,122.1,55.12,60.4,45.64,67.09,NaN,NaN,37.23,77.54
29949,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36.31,NaN,NaN,NaN
105372878,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,162.40,NaN,NaN,NaN
8291,133.70,131.90,179.3,152.3,52.89,173.9,109.50,48.28,209.30,170.1,125.00,148.80
200403,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.90


In [30]:
df_numeric["total_expression"] = df_numeric.sum(axis=1)

top_genes = df_numeric.sort_values("total_expression", ascending=False)

top_genes.head(20)

,GSM5548014,GSM5548015,GSM5548016,GSM5548017,GSM5548018,GSM5548019,GSM5548020,GSM5548021,GSM5548022,GSM5548023,GSM5548024,GSM5548025,total_expression
GeneID,,,,,,,,,,,,,
23025,999400.00,999400.00,998100.00,999100.000,999100.00,996900.00,961900.00,996300.00,998800.00,997300.00,992500.00,997000.00,1.193580e+07
5225,0.00,0.00,0.00,0.000,0.00,0.00,32310.00,0.00,0.00,0.00,0.00,0.00,3.231000e+04
4625,22.01,19.31,67.50,30.870,69.68,0.00,3116.00,1950.00,137.90,40.74,4706.00,882.10,1.104211e+04
3757,0.00,36.38,0.00,29.080,0.00,0.00,896.80,858.80,18.56,172.70,1817.00,530.90,4.360220e+03
5133,60.41,0.00,92.64,127.100,382.50,157.20,237.60,0.00,324.40,111.80,322.90,201.80,2.018350e+03
161159,0.00,0.00,0.00,0.000,0.00,1541.00,0.00,0.00,0.00,0.00,0.00,247.20,1.788200e+03
9759,42.12,124.70,72.65,328.600,195.80,0.00,138.00,415.80,14.14,48.72,0.00,369.20,1.749730e+03
9625,0.00,21.19,1222.00,0.000,0.00,460.80,0.00,0.00,0.00,0.00,0.00,0.00,1.703990e+03
8291,133.70,131.90,179.30,152.300,52.89,173.90,109.50,48.28,209.30,170.10,125.00,148.80,1.634970e+03
